# Backtrader is a live Trading and backtesting platform written in Python.

Source : https://github.com/mementum/backtrader

## 1.Initialize Backtrater

In [ ]:
pip install backtrader

2. Create a simple account

In [ ]:
import backtrader as bt

cerebro = bt.Cerebro() # create a "Cerebro" engine instance to run the strategy.
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())


Cerebro is the core component of the backtester and is responsible for managing the entire backtesting process. It acts as a container for strategies, data feeds, analyzers, and other components necessary for backtesting.

3. Add a data feed
Now lets add a Data Feed.

We will use our yfinance package (yahoo API) to get the required data for a specific ticker


In [ ]:

# Access yfinance and call get_data() method to download the data

# Import the yfinance library
import yfinance as yf
import pandas as pd

# Function to get the ticker data
def get_ticker(stock_symbol, timespan, time):
    # Fetch data from yfinance
    ticker = yf.Ticker(stock_symbol)
    # Fetch historical data from yfinance
    data = ticker.history(period=timespan, interval=time)
    return data

# Example: Get the data for the stock symbol 'AAPL' for the last 2 years with 1 day interval
data = get_ticker('AAPL', '2y', '1d')

# Convert the yfinance DataFrame to a Backtrader data feed
class PandasData(bt.feeds.PandasData):
    # Define the columns
    params = (
        ('datetime', None),
        ('open', 'Open'),
        ('high', 'High'),
        ('low', 'Low'),
        ('close', 'Close'),
        ('volume', 'Volume'),
        ('openinterest', None),
    )


# Add the data feed to Cerebro
data_feed = PandasData(dataname=data)

# Add the Data Feed to Cerebro
cerebro.adddata(data_feed)

## 2. Create a simple strategy 
Great! Now that the account has some cash in and the data is already fed, we need to create our strategy. 

We will start with a foolish strastegy of buying the ticker if it was down the past three days! 

In [ ]:
# Create a Stratey
class BuyStrategy(bt.Strategy):

    def log(self, txt, dt=None):
        ''' Logging function fot this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.datas[0].close

    def next(self):
        # Simply log the closing price of the series from the reference
        #self.log('Close, %.2f' % self.dataclose[0])

        if self.dataclose[0] < self.dataclose[-1]:
            # current close less than previous close

            if self.dataclose[-1] < self.dataclose[-2]:
                # previous close less than the previous close

                # BUY, BUY, BUY!!! (with all possible default parameters)
                self.log('BUY CREATE, %.2f' % self.dataclose[0])
                self.buy()

Now lets add it to our cerebro!

In [ ]:

cerebro = bt.Cerebro() # create a "Cerebro" engine instance to run the strategy.
# Add the data feed to Cerebro
cerebro.adddata(data_feed)
# Add the strategy to Cerebro
cerebro.addstrategy(BuyStrategy)
# Print out the starting conditions
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
# Run over everything
cerebro.run()
# Plot the results
fig = cerebro.plot(iplot=False, volume=False)[0][0]

print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())


## 3. Expand strategy to also sell

We were lucky that our stock performed well last couple of years! But now we can only buy somethingQ

Let's expand the algorithm to also sell!!

In [ ]:
# Create a Stratey
class BuySellStrategy(bt.Strategy):

    def log(self, txt, dt=None):
        ''' Logging function fot this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.datas[0].close

        # To keep track of pending orders
        self.order = None

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log('BUY EXECUTED, %.2f' % order.executed.price)
            elif order.issell():
                self.log('SELL EXECUTED, %.2f' % order.executed.price)

            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')

        # Write down: no pending order
        self.order = None

    def next(self):
        # Simply log the closing price of the series from the reference
        #self.log('Close, %.2f' % self.dataclose[0])

        # Check if an order is pending ... if yes, we cannot send a 2nd one
        if self.order:
            return

        # Check if we are in the market
        if not self.position:

            # Not yet ... we MIGHT BUY if ...
            if self.dataclose[0] < self.dataclose[-1]:
                    # current close less than previous close

                    if self.dataclose[-1] < self.dataclose[-2]:
                        # previous close less than the previous close

                        # BUY, BUY, BUY!!! (with default parameters)
                        self.log('BUY CREATE, %.2f' % self.dataclose[0])

                        # Keep track of the created order to avoid a 2nd order
                        self.order = self.buy()

        else:

            # Already in the market ... we might sell
            if len(self) >= (self.bar_executed + 5):
                # SELL, SELL, SELL!!! (with all possible default parameters)
                self.log('SELL CREATE, %.2f' % self.dataclose[0])

                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()


In [ ]:
cerebro = bt.Cerebro() # create a "Cerebro" engine instance to run the strategy.
# Add the data feed to Cerebro
cerebro.adddata(data_feed)
# Print out the starting conditions
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())


cerebro.addstrategy(BuySellStrategy)
# Print out the starting conditions
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
# Run over everything
cerebro.run()
# Plot the results
fig = cerebro.plot(iplot=False, volume=False)[0][0]

print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue()) 

## 4. Switch strategy and utilize indicators
We made less money, but at least now we can also short a stock! Now lets implement a simple SMA indicator to buy/sell


In [ ]:
from datetime import datetime
import backtrader as bt
import matplotlib.pyplot as plt


class SmaCross(bt.SignalStrategy):
    def __init__(self):
        sma1, sma2 = bt.ind.SMA(period=10), bt.ind.SMA(period=30)
        crossover = bt.ind.CrossOver(sma1, sma2)
        self.signal_add(bt.SIGNAL_LONG, crossover)


cerebro = bt.Cerebro() # create a "Cerebro" engine instance to run the strategy.
# Add the data feed to Cerebro
cerebro.adddata(data_feed)

# Print out the starting conditions
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

cerebro.addstrategy(SmaCross)
cerebro.run()
# Plot the results
fig = cerebro.plot(iplot=False, volume=False)[0][0]

print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())
